<a href="https://colab.research.google.com/github/TU-USUARIO/labo1-colabs/blob/main/03_Estadistica_gaussiana_y_compatibilidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 03 — Estadística de una variable, la gaussiana y la compatibilidad

**Laboratorio 1 · Clase 3**

**Objetivos.**

1. Justificar el promedio como **estimador**, y no como procedimiento (O3.1).
2. Calcular la desviación estándar sin el error silencioso de NumPy (O3.2).
3. Distinguir $s$ de SEM y decidir cuál reportar (O3.3).
4. Elegir el ancho de bin con criterio declarado (O3.4).
5. Verificar que SEM cae como $1/\sqrt{N}$ — y descubrir que **medir mejor cae como $1/N$** (O3.5).
6. Superponer la gaussiana **predicha**, y entender por qué no la ajustamos (O3.6).
7. Reconocer cuándo esperar una gaussiana y cuándo no, y saldar la deuda del Colab 02:
   $\sigma = \Delta/\sqrt{12}$ (O3.7).
8. Decidir compatibilidad entre dos mediciones (O3.8) y aplicar Chauvenet con criterio (O3.9).
9. Ver que **promediar enmascara el sistemático** (O3.10).

**Requisitos previos:** Colabs 01 y 02.

Es el notebook más largo del curso y la clase conceptualmente más importante. Si algo de acá queda
flojo, arrastra hasta la última práctica.

> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import floor, log10

rng = np.random.default_rng(20260826)

def reportar(x, dx, unidad=""):
    '''Del Colab 02: escribe (valor ± error) con las cifras correctas.'''
    orden   = floor(log10(abs(dx)))
    cifras  = 2 if int(dx / 10**orden) in (1, 2) else 1
    dec     = max(-(orden - (cifras - 1)), 0)
    return f"({x:.{dec}f} ± {dx:.{dec}f}) {unidad}".strip()

---
## 1. Los dos experimentos del día

Hoy medimos dos tiempos, y la comparación entre ellos es la mitad de la clase.

**El faro.** Un LED que se enciende con período fijo, del orden de 1 a 2 s. Tiene una propiedad que
ningún otro sistema del curso tiene: **su período verdadero no cambia**. Toda la dispersión que
observes es tuya, del observador. Eso lo convierte en el mejor instrumento posible para estudiar el
error de medición aislado.

**El tiempo de reacción.** Regla en caída libre. Acá el sistema **sí** tiene dispersión propia: tu
reacción varía de intento a intento. Y hay un detalle: lo que medís es una **distancia**, y el tiempo
sale de $t = \sqrt{2d/g}$. O sea que hay una propagación en el medio — que ya sabés hacer.

Las celdas de abajo generan datos de ejemplo con la estructura correcta. **Cuando tengas los tuyos,
reemplazalas por `np.loadtxt`.**

In [ ]:
# ============ DATOS DE EJEMPLO — REEMPLAZAR POR LOS PROPIOS ============
# FARO: para cada tanda de n destellos, el tiempo total cronometrado.
# La clave física: el error está en los dos apretones del cronómetro,
# así que NO crece con la duración de la tanda.
T0_verdadero = 1.437          # s   (no lo conocés en la vida real)
sigma_evento = 0.17           # s   dispersión del cronometrado, independiente de n

faro = {}
for n, repeticiones in [(1, 50), (10, 8), (20, 8), (50, 8)]:
    faro[n] = n * T0_verdadero + rng.normal(0, sigma_evento, repeticiones)
    np.savetxt(f'faro_n{n}.txt', faro[n], fmt='%.3f')

# TIEMPO DE REACCIÓN: se mide la distancia de caída de la regla, en metros
g = 9.81
t_fisiologico = 0.150 + rng.gamma(shape=4.0, scale=0.018, size=120)   # asimétrico a propósito
d = g * t_fisiologico**2 / 2
np.savetxt('caida_regla.txt', d, fmt='%.4f')
# =======================================================================

d = np.loadtxt('caida_regla.txt')          # m
t = np.sqrt(2 * d / g)                     # s  <- medición indirecta (Colab 02)

print(f"Faro: tandas de n =", list(faro.keys()))
for n, x in faro.items():
    print(f"   n = {n:2d}: {len(x):2d} tandas, tiempo total medio = {x.mean():7.3f} s")
print(f"\nRegla: {len(d)} caídas, distancia media = {d.mean()*100:.1f} cm "
      f"-> t medio = {t.mean():.4f} s")

> **Nota sobre la incerteza de la regla.** La resolución es 1 mm, así que $\sigma_d = \Delta/\sqrt{12}
> \approx 0{,}29$ mm. Propagando, $\sigma_t = \sigma_d/(g\,t) \approx 1{,}3\times10^{-4}$ s. Compará ese
> número con la dispersión de tus tiempos, que va a ser del orden de 0,03 s. **El instrumento no es el
> límite acá: sos vos.** Ésa es exactamente la situación en que la incerteza deja de ser una propiedad
> del instrumento, y por eso hace falta todo lo que sigue.

---
## 2. El promedio no es una receta: es un estimador

Escribimos

$$\bar{x} = \frac{1}{N}\sum_{i=1}^{N} x_i$$

y lo llamamos "el promedio". Pero la pregunta relevante no es cómo se calcula sino **qué estima**.

Detrás de tus $N$ mediciones hay una distribución de probabilidad: el proceso de medir tiene un valor
esperado $\mu$ y una dispersión $\sigma$. Cada medición individual es una muestra de esa
distribución. El promedio es un **estimador de $\mu$**, y bajo tres supuestos es el mejor disponible:

1. los errores son **puramente aleatorios** (no hay sesgo sistemático);
2. las mediciones son **independientes** entre sí;
3. la distribución tiene **varianza finita**.

Si el supuesto 1 falla, el promedio converge — pero converge al valor equivocado, y como la
dispersión sigue bajando, el resultado *parece* excelente. Volvemos sobre eso en la Sección 9.

Si falla el 2 (te fuiste cansando, el instrumento derivó), el SEM subestima la incerteza real.

In [ ]:
N = len(t)
promedio = np.mean(t)
print(f"np.mean  : {promedio:.6f} s")
print(f"suma / N : {np.sum(t)/N:.6f} s")
print(f"N = {N}")

---
## 3. Desviación estándar: el error silencioso de NumPy

La desviación estándar **muestral** es

$$s = \sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(x_i - \bar{x})^2}$$

Prestá atención al $N-1$ (**corrección de Bessel**). El motivo: para calcular la dispersión respecto
de $\bar{x}$ ya "gastaste" un grado de libertad estimando $\bar{x}$ con los mismos datos. Dividir por
$N$ subestima sistemáticamente la dispersión.

**El problema:** `np.std()` usa por defecto `ddof=0`, es decir, divide por $N$. Calcula la desviación
estándar *poblacional*, que no es la que corresponde a un conjunto de mediciones experimentales.
No da error. No avisa. Devuelve un número un poco chico. **Siempre escribí `ddof=1`.**

In [ ]:
s_mal  = np.std(t)              # ddof=0 por defecto  -> POBLACIONAL (incorrecto acá)
s_bien = np.std(t, ddof=1)      # muestral            -> correcto

print(f"np.std(t)          = {s_mal:.6f} s   <- NO usar")
print(f"np.std(t, ddof=1)  = {s_bien:.6f} s   <- usar ésta")
print(f"diferencia relativa: {100*(s_bien-s_mal)/s_bien:.3f} %")
print()
for n in [3, 5, 10, 30, 100, 1000]:
    factor = np.sqrt(n / (n - 1))
    print(f"N = {n:5d}  ->  s_muestral / s_poblacional = {factor:.4f}"
          f"   ({100*(factor-1):5.2f} % de subestimación si te olvidás)")

Con $N=3$ —el caso típico de "medí tres veces"— olvidarse de `ddof=1` subestima la dispersión en un
22 %. Con $N=1000$ el efecto es despreciable, pero para entonces ya instalaste la idea equivocada.

> **Ejercicio 3.1.** Auditá cualquier código de análisis que ya tengas (tuyo o de un compañero)
> buscando `np.std(` sin `ddof=1`. Es la clase de error que sobrevive años sin que nadie lo note.

---
## 4. Desviación estándar contra error de la media

Son dos cosas distintas y responden a dos preguntas distintas:

| Cantidad | Fórmula | Responde a |
|---|---|---|
| Desviación estándar $s$ | $\sqrt{\frac{1}{N-1}\sum(x_i-\bar{x})^2}$ | ¿Cuánto se dispersa **una medición individual**? |
| Error de la media | $\mathrm{SEM} = s/\sqrt{N}$ | ¿Cuánta incerteza tiene **el promedio**? |

> **La desviación estándar no disminuye al aumentar $N$** — es una propiedad del proceso de medición.
> **El error de la media sí disminuye**, como $1/\sqrt{N}$ — porque promediar más mediciones mejora la
> estimación del valor central, no la calidad de cada medición individual.

Cuál de las dos reportás depende de qué estés informando. Si el resultado de tu experimento es el
promedio, la incerteza que corresponde es el SEM. Si querés describir cuánto varía tu tiempo de
reacción entre intentos, la que corresponde es $s$.

In [ ]:
sem = s_bien / np.sqrt(N)
print(f"x̄   = {promedio:.5f} s")
print(f"s   = {s_bien:.5f} s      (dispersión de UNA medición)")
print(f"SEM = {sem:.5f} s      (incerteza DEL PROMEDIO)")
print(f"s/SEM = {s_bien/sem:.2f}   y  √N = {np.sqrt(N):.2f}")
print()
print("Si informás tu tiempo de reacción típico :", reportar(promedio, sem, "s"), " (SEM)")
print("Si informás cuánto varía entre intentos  :", reportar(promedio, s_bien, "s"), " (s)")

In [ ]:
# Verificación empírica: en vez de creerle a la fórmula, la miramos.
ns = np.arange(2, N + 1)
s_acum   = np.array([np.std(t[:n], ddof=1) for n in ns])
sem_acum = s_acum / np.sqrt(ns)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(ns, s_acum,   'o-', ms=3, lw=1, label='$s$  (desviación estándar)')
ax.plot(ns, sem_acum, 's-', ms=3, lw=1, label='SEM $= s/\\sqrt{N}$')
ax.plot(ns, s_bien / np.sqrt(ns), 'k--', lw=1, label='$s_{final}/\\sqrt{N}$ (teórico)')
ax.set_xlabel('Cantidad de mediciones usadas $N$')
ax.set_ylabel('Dispersión [s]')
ax.set_title('$s$ se estabiliza; el error de la media cae como $1/\\sqrt{N}$')
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

La curva de $s$ fluctúa al principio y después **se aplana**. La del SEM **sigue bajando**. Ésa es
toda la diferencia entre las dos cantidades, en una figura.

> **Ejercicio 3.2.** ¿Cuántas mediciones necesitarías para reducir el SEM a la mitad del valor que
> tiene ahora? ¿Y a la décima parte? ¿Te parece razonable el costo experimental?

---
## 5. El histograma y el ancho de bin

El histograma es la primera mirada a la forma de la distribución. Pero el resultado depende del ancho
de bin, y elegirlo "hasta que se vea lindo" es una decisión arbitraria que puede cambiar la
conclusión.

Existen criterios objetivos. El más simple es la **regla de Scott** (Scott, *Biometrika* **66**(3),
605–610, 1979):

$$ h = \frac{3{,}49\, s}{N^{1/3}} $$

que minimiza el error cuadrático medio integrado suponiendo distribución aproximadamente normal. Una
alternativa más robusta frente a valores atípicos es **Freedman–Diaconis** (*Z. Wahrsch. Verw.
Gebiete* **57**, 453–476, 1981), que usa el rango intercuartílico en lugar de $s$.

In [ ]:
def bins_scott(x):
    x = np.asarray(x); n = len(x)
    h = 3.49 * np.std(x, ddof=1) / n**(1/3)
    return max(1, int(np.ceil((x.max() - x.min()) / h)))

def bins_freedman_diaconis(x):
    x = np.asarray(x); n = len(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    h = 2 * iqr / n**(1/3)
    return max(1, int(np.ceil((x.max() - x.min()) / h)))

print("Scott            :", bins_scott(t), "bins")
print("Freedman-Diaconis:", bins_freedman_diaconis(t), "bins")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, nb, titulo in zip(axes, [4, bins_scott(t), 60],
                          ['4 bins (demasiado grueso)',
                           f'{bins_scott(t)} bins (regla de Scott)',
                           '60 bins (demasiado fino)']):
    ax.hist(t, bins=nb, edgecolor='k', alpha=0.75)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel('Tiempo de reacción [s]'); ax.grid(alpha=0.3)
axes[0].set_ylabel('Frecuencia')
fig.tight_layout(); plt.show()

Los tres histogramas son de **los mismos datos**. El primero no deja ver si la distribución es
simétrica; el tercero muestra estructura que no existe. Por eso el criterio de bin se declara en el
informe, igual que cualquier otra decisión de análisis.

---
## 6. El punto de la clase: $1/\sqrt{N}$ contra $1/N$

Acá entra el faro, y acá está la diferencia entre **repetir** y **medir mejor**.

Cronometrás una tanda de $n$ destellos. El error está en los dos apretones del cronómetro, así que la
incerteza del **tiempo total** no depende de $n$: es siempre $\sigma_{\text{ev}}$. Pero el período es
$T = t_{\text{total}}/n$, y por lo tanto

$$ \sigma_T = \frac{\sigma_{\text{ev}}}{n} $$

Eso **cae como $1/n$**. Mientras tanto, promediar $m$ mediciones de un destello da
$\sigma_T = \sigma_{\text{ev}}/\sqrt{m}$, que cae como $1/\sqrt{m}$.

Primero verifiquemos la afirmación de la que todo depende: que la dispersión del tiempo total no
crece con $n$.

In [ ]:
print(f"{'n':>4s} {'tandas':>7s} {'t_total medio':>14s} {'s(t_total)':>12s} "
      f"{'σ_s/s':>8s} {'T = t/n':>10s}")
print("-" * 60)
n_vals, s_tot_vals, m_reps = [], [], []
for n, x in sorted(faro.items()):
    s_tot = np.std(x, ddof=1)
    m = len(x)
    incert_de_s = 1 / np.sqrt(2 * (m - 1))      # incerteza RELATIVA de s con m datos
    n_vals.append(n); s_tot_vals.append(s_tot); m_reps.append(m)
    print(f"{n:4d} {m:7d} {x.mean():14.3f} {s_tot:12.4f} "
          f"{100*incert_de_s:7.0f}% {x.mean()/n:10.4f}")

n_vals   = np.array(n_vals)
s_tot_vals = np.array(s_tot_vals)
m_reps   = np.array(m_reps)

# Estimador combinado (pooled) de σ_evento: como la dispersión NO depende de n,
# conviene juntar toda la información en lugar de usar una sola tanda.
gl = m_reps - 1
sigma_ev = np.sqrt(np.sum(gl * s_tot_vals**2) / np.sum(gl))
print(f"\nσ_evento combinado sobre los {gl.sum()} grados de libertad: {sigma_ev:.4f} s")
print(f"(incerteza relativa de esta estimación: {100/np.sqrt(2*gl.sum()):.0f} %)")

sigmaT_vals = sigma_ev / n_vals

La columna `s(t_total)` es aproximadamente constante: **no crece con $n$**. Ésa es la observación
experimental clave, y no es obvia — habría que verificarla en cualquier experimento nuevo antes de
usarla.

**Pero no la mires sin la columna de al lado.** Una desviación estándar estimada con $m$ datos tiene
ella misma una incerteza relativa de aproximadamente

$$ \frac{\sigma_s}{s} \approx \frac{1}{\sqrt{2(m-1)}} $$

Con $m = 8$ tandas, eso es un **27 %**. Así que dos valores de `s(t_total)` que difieran en un 30 %
son perfectamente compatibles entre sí, y sería un error leer esas diferencias como si fueran
señal. Por eso, y porque la hipótesis es que $\sigma$ no depende de $n$, el código combina todos los
grados de libertad en una sola estimación de $\sigma_{\text{ev}}$ — que es lo correcto y además es
mucho más estable.

> **Esto es una lección por derecho propio:** $s$ es un número medido, con su propia incerteza, y
> con pocos datos esa incerteza es enorme. Cuando alguien dice "medí tres veces y me dio $s = 0{,}2$",
> lo que en realidad tiene es $s = 0{,}2 \pm 0{,}1$.

In [ ]:
# Los dos métodos, en un solo gráfico log-log
m_vals = np.arange(1, len(faro[1]) + 1)
sem_metodoA = sigma_ev / np.sqrt(m_vals)  # promediar m mediciones de UN destello

fig, ax = plt.subplots(figsize=(7, 4.6))
ax.loglog(m_vals, sem_metodoA, 'o-', ms=3, lw=1,
          label='A: promediar $m$ mediciones de 1 destello   ($\\propto m^{-1/2}$)')
ax.loglog(n_vals, sigmaT_vals, 's-', ms=7, lw=1.4, color='crimson',
          label='B: cronometrar una tanda de $n$ destellos   ($\\propto n^{-1}$)')
ax.loglog(m_vals, sigma_ev / m_vals, 'k:', lw=1, label='pendiente $-1$ (referencia)')
ax.set_xlabel('Cantidad de destellos observados')
ax.set_ylabel('Incerteza del período $\\sigma_T$ [s]')
ax.set_title('Mismo tiempo de mesada, dos métodos, un orden de magnitud de diferencia')
ax.grid(alpha=0.3, which='both'); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

# La comparación justa: mismo número de destellos observados
n_ref = 50
sA = sigma_ev / np.sqrt(n_ref)
sB = sigma_ev / n_ref
print(f"Observando {n_ref} destellos en total (mismo tiempo de mesada):")
print(f"   método A ({n_ref} mediciones de 1 destello): σ_T = {sA:.5f} s")
print(f"   método B (1 tanda de {n_ref} destellos)    : σ_T = {sB:.5f} s")
print(f"   B es {sA/sB:.1f} veces mejor   (y √{n_ref} = {np.sqrt(n_ref):.1f})")

**Ésta es la respuesta al título de la clase.** Los dos métodos observan la misma cantidad de
destellos y ocupan el mismo tiempo de mesada. Uno da un resultado siete veces mejor que el otro.

La conclusión no es estadística, es de **diseño experimental**: repetir mejora como $1/\sqrt{N}$ y se
agota rápido; cambiar el método puede mejorar como quieras. En algún punto conviene dejar de repetir
y empezar a medir mejor.

Eso es exactamente lo que hacemos en la Clase 4, donde el photogate reemplaza al pulgar y la mejora
es otro factor grande.

> **Ejercicio 3.3.** ¿Cuántas mediciones de un destello harían falta para igualar la incerteza de una
> sola tanda de 50? ¿Cuánto tiempo de mesada es eso? Ahora respondé la pregunta incómoda: ¿por qué
> entonces medimos también las 50 individuales, si sabemos de antemano que es el método peor?

In [ ]:
# El resultado del faro, con el mejor método
T_faro = faro[50].mean() / 50
sig_T_faro = sigma_ev / (50 * np.sqrt(len(faro[50])))   # tanda de 50, promediada sobre las repeticiones
print("Período del faro:", reportar(T_faro, sig_T_faro, "s"))

> **Atención al último paso.** Acá combinamos las dos ideas: la tanda de 50 divide el error por 50,
> y promediar las 8 tandas lo divide por $\sqrt{8}$ más. Las dos mejoras se multiplican porque son
> independientes. Fijate que la incerteza que usamos es el **error de la media de las ocho tandas**,
> no la dispersión de una sola.

---
## 7. La gaussiana, el TCL y el error de resolución

$$ f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left[-\frac{(x-\mu)^2}{2\sigma^2}\right] $$

Dos parámetros y ningún otro: el centro $\mu$ y el ancho $\sigma$.

In [ ]:
def gaussiana(x, mu, sigma):
    return np.exp(-(x - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))

from scipy.integrate import quad
for k in [1, 2, 3]:
    p, _ = quad(gaussiana, -k, k, args=(0, 1))
    print(f"±{k}σ  ->  {100*p:.2f} % de la probabilidad")

Esos números son la razón por la que un resultado a $3\sigma$ de lo esperado llama la atención: si el
modelo fuera correcto, ocurriría en un 0,3 % de los casos. Ojo con el abuso: vale **si** la
distribución es normal y **si** no hay sistemáticos. Las dos condiciones fallan seguido.

### Predecir, no ajustar

Podríamos *ajustar* una gaussiana al histograma y ver qué $\mu$ y $\sigma$ salen. Pero es más
informativo hacer lo contrario: tomar $\bar{x}$ y $s$ —que ya calculaste, sin ninguna hipótesis sobre
la forma de la distribución— y **predecir** con ellos la curva. Después mirás si describe los datos.

Si describe bien, tus estimadores capturan la distribución. Si no, aprendiste algo real sobre tu
proceso de medición. Ajustar la curva te habría dado siempre "la mejor gaussiana posible", incluso
cuando ninguna gaussiana sirve.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.hist(t, bins=bins_scott(t), density=True, edgecolor='k', alpha=0.65,
        label='datos (normalizados)')
xx = np.linspace(t.min() - 3*s_bien, t.max() + 3*s_bien, 400)
ax.plot(xx, gaussiana(xx, promedio, s_bien), 'crimson', lw=2,
        label='gaussiana PREDICHA con $\\bar{x}$ y $s$')
ax.set_xlabel('Tiempo de reacción [s]'); ax.set_ylabel('Densidad de probabilidad')
ax.set_title('La curva no está ajustada: está predicha a partir de los estimadores')
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

`density=True` es imprescindible acá: normaliza el área a 1 para que las escalas sean comparables.

**Mirá las colas.** En tiempos de reacción es habitual una cola hacia valores altos —hay un piso
fisiológico y no hay techo— que rompe la simetría. Si tus datos la tienen, la gaussiana predicha va a
quedar corrida y demasiado angosta. Eso no es un error del análisis: es un resultado.

### Teorema Central del Límite

¿Por qué aparece la gaussiana en todos lados? Porque los errores rara vez tienen una sola causa: son
la suma de muchas contribuciones pequeñas e independientes. El TCL dice que esa suma tiende a una
gaussiana, cualquiera sea la distribución de cada término.

In [ ]:
M = 20000
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, k in zip(axes, [1, 2, 5, 12]):
    suma = rng.uniform(-1, 1, size=(M, k)).sum(axis=1)
    ax.hist(suma, bins=60, density=True, edgecolor='none', alpha=0.75)
    xx = np.linspace(suma.min(), suma.max(), 300)
    ax.plot(xx, gaussiana(xx, 0, np.sqrt(k/3)), 'crimson', lw=1.8)
    ax.set_title(f'suma de k = {k}', fontsize=10)
    ax.set_xlabel('valor'); ax.grid(alpha=0.3)
axes[0].set_ylabel('densidad')
fig.suptitle('De uniforme a gaussiana: el Teorema Central del Límite', y=1.04)
fig.tight_layout(); plt.show()

Con $k=1$ es un rectángulo. Con $k=2$ ya es un triángulo. Con $k=12$ es indistinguible de una
gaussiana a ojo. Y nunca supusimos normalidad en ningún lado.

**El corolario, que casi nunca se enseña:** si el error está dominado por **una sola** fuente, la
distribución no tiene por qué ser gaussiana. El caso $k=1$ de este gráfico es exactamente el error
de resolución del Colab 01: uniforme, no gaussiano.

Y acá se salda la deuda del Colab 02. La varianza de una uniforme de ancho $\Delta$ es
$\Delta^2/12$, así que

$$ \sigma = \frac{\Delta}{\sqrt{12}} \approx 0{,}29\,\Delta $$

Ése es el $\sigma$ que corresponde cuando la resolución del instrumento es la única fuente de
incerteza.

> **Nota.** Acá usamos números aleatorios sólo como recurso ilustrativo. No es una simulación Monte
> Carlo como técnica de análisis de datos: eso es otra cosa y no forma parte de este curso.

In [ ]:
# Verificación directa de Δ/√12
for delta in [1.0, 0.05, 0.01]:
    u = rng.uniform(-delta/2, delta/2, 200000)
    print(f"Δ = {delta:5.2f}   ->  s medida = {u.std(ddof=1):.6f}   "
          f"Δ/√12 = {delta/np.sqrt(12):.6f}")

---
## 8. ¿Dos mediciones son compatibles?

Ésta es la pregunta que aparece cada vez que medís algo por dos métodos, o cuando comparás tu
resultado con el valor de tabla. La respuesta **no** es "sí, dan parecido".

$$ z = \frac{|x_1 - x_2|}{\sqrt{\sigma_1^2 + \sigma_2^2}} $$

Es la diferencia medida en unidades de la incerteza **de la diferencia**. Criterio operativo:

| $z$ | Lectura |
|---|---|
| $z < 1$ | compatibles; la diferencia es menor que el ruido |
| $1 \le z < 2$ | compatibles dentro de lo esperable |
| $2 \le z < 3$ | en tensión; conviene revisar |
| $z \ge 3$ | incompatibles: o hay un sistemático, o alguna incerteza está subestimada |

El caso más interesante es el último, y casi nunca significa que la física esté mal: casi siempre
significa que alguien subestimó su error.

**Importante:** el test supone mediciones **independientes**. Comparar dos análisis distintos del
mismo conjunto de datos con esta fórmula está mal, y volvemos sobre eso en la Clase 6.

In [ ]:
def compatibilidad(x1, dx1, x2, dx2, etiquetas=('medición 1', 'medición 2')):
    z = abs(x1 - x2) / np.sqrt(dx1**2 + dx2**2)
    if   z < 1: veredicto = "compatibles"
    elif z < 2: veredicto = "compatibles dentro de lo esperable"
    elif z < 3: veredicto = "en tensión — revisar"
    else:       veredicto = "INCOMPATIBLES — sistemático o incerteza subestimada"
    print(f"{etiquetas[0]}: {x1:.5g} ± {dx1:.2g}")
    print(f"{etiquetas[1]}: {x2:.5g} ± {dx2:.2g}")
    print(f"z = {z:.2f}  ->  {veredicto}\n")
    return z

# Los dos métodos del faro: ¿dan lo mismo?
T_A = faro[1].mean();  sig_A = np.std(faro[1], ddof=1) / np.sqrt(len(faro[1]))
_ = compatibilidad(T_A, sig_A, T_faro, sig_T_faro,
                   ('faro, 50 mediciones de 1 destello', 'faro, 8 tandas de 50'))

# Un caso con sistemático
_ = compatibilidad(9.62, 0.02, 9.81, 0.03, ('g con roce no modelado', 'valor de referencia'))

El segundo caso da $z \approx 5$. Ningún aumento de $N$ lo va a arreglar: es un error sistemático, y
se corrige mejorando el experimento o modelando el efecto, no midiendo más veces.

> **Nota.** Si los dos métodos del faro dan compatibles, ¿se pueden **combinar** en un solo número?
> Sí, y la herramienta es el promedio ponderado — que en este curso aparece en la Clase 6, porque es
> un caso particular del ajuste ponderado y conviene verlos juntos.

---
## 9. Datos anómalos: el criterio de Chauvenet

Un dato que se escapa es **primero** una hipótesis sobre un error identificable (te distrajiste,
alguien golpeó la mesa, anotaste mal) y **solo después** un problema estadístico. Si podés
identificar la causa, lo descartás y lo informás. Si no podés, entra Chauvenet.

**El criterio:** con $N$ mediciones, se descarta el dato cuyo $z = |x_i - \bar{x}|/s$ cumpla

$$ N \cdot P(|Z| > z) < 0{,}5 $$

es decir, si el número **esperado** de datos tan extremos como ése, en una muestra de tamaño $N$, es
menor que medio dato.

**Las tres reglas de conducta, que importan más que la fórmula:**

1. El criterio se fija **antes** de mirar los datos.
2. Se aplica **una sola vez**. Nada de recalcular $\bar{x}$ y volver a podar hasta que quede lindo.
3. **Todo descarte se informa**, con el criterio y la cantidad de puntos eliminados.

In [ ]:
from scipy.stats import norm

def chauvenet(x, verbose=True):
    '''Devuelve la máscara de datos que SOBREVIVEN al criterio de Chauvenet.'''
    x = np.asarray(x)
    n = len(x)
    z = np.abs(x - x.mean()) / np.std(x, ddof=1)
    esperados = n * 2 * norm.sf(z)          # cantidad esperada de datos tan extremos
    mantener = esperados >= 0.5
    if verbose:
        for i in np.where(~mantener)[0]:
            print(f"  descartar x[{i}] = {x[i]:.4f}   z = {z[i]:.2f}   "
                  f"esperados = {esperados[i]:.3f}")
        print(f"  sobreviven {mantener.sum()} de {n} datos")
    return mantener

# Metemos a mano un dato anómalo para ver el criterio en acción
t_con_outlier = np.append(t, 0.62)      # una distracción
print("Sobre los datos con un anómalo agregado:")
mask = chauvenet(t_con_outlier)

print()
print(f"con el anómalo : {reportar(t_con_outlier.mean(), np.std(t_con_outlier, ddof=1)/np.sqrt(len(t_con_outlier)), 's')}")
tt = t_con_outlier[mask]
print(f"sin el anómalo : {reportar(tt.mean(), np.std(tt, ddof=1)/np.sqrt(len(tt)), 's')}")

> **Ejercicio 3.4.** Aplicá `chauvenet()` a tus datos reales **sin** agregar nada. Si descarta algún
> punto, andá al cuaderno y buscá si hay una causa anotada. Si no descarta ninguno, no fuerces el
> criterio.
>
> **Ejercicio 3.5.** *(conceptual)* Chauvenet supone que los datos son gaussianos. Tu histograma de
> tiempos de reacción probablemente no lo sea. ¿En qué dirección se equivoca el criterio si la
> distribución tiene una cola hacia arriba? ¿Descarta de más o de menos, y de qué lado?

---
## 10. Promediar no corrige un sistemático: lo enmascara

Ésta es la contracara estadística de lo que viste en el Colab 02, y es peor de lo que parece.

Al soltar la regla hay un retardo: soltás un instante después de decidir soltar. Eso agrega un sesgo
sistemático a **todas** tus mediciones por igual. Simulemos dos observadores midiendo el mismo tiempo
de reacción verdadero:

- **A:** ruidoso pero sin sesgo.
- **B:** muy poco ruidoso pero con un retardo de $+15$ ms.

In [ ]:
VERDADERO = 0.220        # s
N_max = 2000
ns2 = np.arange(1, N_max + 1)

A = rng.normal(VERDADERO,         0.030, N_max)   # ruidoso, insesgado
B = rng.normal(VERDADERO + 0.015, 0.008, N_max)   # preciso, sesgado

mediaA = np.cumsum(A) / ns2
mediaB = np.cumsum(B) / ns2

fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.axhline(VERDADERO, color='k', ls='--', lw=1.2, label='valor verdadero')
ax.plot(ns2, mediaA, lw=1.2, label='A: ruidoso, sin sesgo')
ax.plot(ns2, mediaB, lw=1.2, color='crimson', label='B: preciso, con sesgo +15 ms')
ax.set_xscale('log')
ax.set_xlabel('Cantidad de mediciones promediadas $N$')
ax.set_ylabel('Promedio acumulado [s]')
ax.set_title('Los dos promedios convergen. Sólo uno converge al valor correcto.')
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

print(f"{'N':>6s} {'A (insesgado)':>26s} {'z_A':>6s} {'B (sesgado)':>26s} {'z_B':>7s}")
print("-" * 76)
for n in [10, 100, 1000, 2000]:
    xa, sa = A[:n].mean(), np.std(A[:n], ddof=1)/np.sqrt(n)
    xb, sb = B[:n].mean(), np.std(B[:n], ddof=1)/np.sqrt(n)
    print(f"{n:6d} {reportar(xa, sa, 's'):>26s} {abs(xa-VERDADERO)/sa:6.1f} "
          f"{reportar(xb, sb, 's'):>26s} {abs(xb-VERDADERO)/sb:7.1f}")

Mirá la columna de B. A medida que $N$ crece, el error de la media baja, el resultado se ve cada vez
**más preciso**, y la discrepancia con el valor verdadero medida en $\sigma$ crece sin parar.

Y mirá también la columna $z_A$, que es donde está la sutileza: el observador **insesgado** también
puede dar un $z$ alto con $N$ chico. Eso es azar, y con $N = 10$ pasa con cierta frecuencia. Lo que
distingue a un sistemático no es que $z$ sea grande en una medición: es que **$z$ crece
sistemáticamente con $N$** mientras el de A fluctúa alrededor de 1. Un solo número de $z$ nunca
alcanza para diagnosticar; hace falta ver cómo se comporta al aumentar la estadística.

Ése es el escenario que hay que aprender a temer: **un resultado que parece cada vez mejor y es cada
vez más falso**. La única defensa es la calibración y la comparación con un método independiente, no
el aumento de $N$.

> **Caso real.** En medición de curvas I–V de junturas con histéresis, promediar la rama de ida con
> la de vuelta da un resultado extraordinariamente reproducible —y sistemáticamente equivocado—
> porque la magnitud que se está promediando depende de la historia del sistema. La reproducibilidad
> no es evidencia de exactitud.

---
## 11. *(Opcional)* Dos histogramas, dos formas

Éste es el contraste que sale gratis de haber medido las dos cosas el mismo día. Es opcional, pero es
el ejercicio que más discusión genera.

- El **faro** tiene período verdadero fijo. El error es de anticipación y se equivoca para los dos
  lados: distribución **simétrica**.
- El **tiempo de reacción** tiene un piso fisiológico y ningún techo: distribución **asimétrica con
  cola hacia arriba**.

Mismo observador, mismo cronómetro, mismo día. Distinta forma. Y de ahí sale la pregunta buena:
cuando la distribución es asimétrica, ¿el promedio sigue siendo el mejor estimador?

In [ ]:
from scipy.stats import skew

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, datos, titulo, unidad in [
        (axes[0], faro[1], 'Faro: 50 cronometrados de 1 destello', 's'),
        (axes[1], t,             'Tiempo de reacción: 120 intentos',     's')]:
    ax.hist(datos, bins=bins_scott(datos), density=True, edgecolor='k', alpha=0.65)
    xx = np.linspace(datos.min(), datos.max(), 300)
    ax.plot(xx, gaussiana(xx, datos.mean(), np.std(datos, ddof=1)), 'crimson', lw=2)
    ax.axvline(np.mean(datos),   color='crimson', ls='--', lw=1.4, label='media')
    ax.axvline(np.median(datos), color='navy',    ls=':',  lw=1.8, label='mediana')
    ax.set_title(f'{titulo}\nasimetría = {skew(datos):+.2f}', fontsize=10)
    ax.set_xlabel(f'[{unidad}]'); ax.grid(alpha=0.3); ax.legend()
axes[0].set_ylabel('Densidad')
fig.tight_layout(); plt.show()

for nombre, datos in [('faro (1 destello)', faro[1]), ('tiempo de reacción', t)]:
    print(f"{nombre:22s}: media = {np.mean(datos):.4f}   "
          f"mediana = {np.median(datos):.4f}   "
          f"diferencia = {100*(np.mean(datos)-np.median(datos))/np.median(datos):+.2f} %")

En una distribución simétrica media y mediana coinciden. En una asimétrica no, y la media se corre
hacia la cola. La pregunta queda planteada y **no la vamos a cerrar acá**: cuál estimador conviene
depende de qué magnitud física querés estimar, y eso no lo decide la estadística.

Lo que sí es una regla firme: **mirá el histograma antes de decidir qué estimador usás**. Reportar
una media sin haber mirado la forma es una apuesta.

---
## 12. Ejercicios

**3.6.** Con tus propios datos, calculá $\bar{x}$, $s$ y SEM del tiempo de reacción. Escribí en una
oración cuál de los dos reportás como incerteza de tu resultado y por qué.

**3.7.** Dividí tus datos en dos mitades (primera y segunda) y calculá $\bar{x}$ y SEM para cada una.
Usá `compatibilidad()`. ¿Son compatibles? Si no lo son, ¿qué podría estar pasando? (Pista: ¿te fuiste
cansando? ¿mejoraste con la práctica?) Éste es un test del supuesto de **independencia**.

**3.8.** Repetí el gráfico de la Sección 4 con los datos **mezclados al azar**
(`rng.permutation(t)`). ¿Cambia la curva de $s$? ¿Y la de SEM? ¿Qué te dice eso?

**3.9.** Repetí el experimento del TCL partiendo de una distribución fuertemente asimétrica
(`rng.exponential`). ¿Cuántos términos hacen falta para que se vea gaussiana? ¿Más o menos que
partiendo de la uniforme?

**3.10.** *(conceptual)* El TCL exige varianza finita. Buscá qué es la distribución de Cauchy y qué
pasa con el promedio de $N$ muestras de una Cauchy cuando $N$ crece. Es el contraejemplo que explica
por qué el supuesto 3 de la Sección 2 está escrito.

**3.11.** Con los datos del faro, estimá $\sigma_{\text{ev}}$ (la incerteza de un cronometrado
individual) de dos maneras independientes: de la dispersión de las 50 mediciones de un destello, y de
la dispersión de las 8 tandas de 50. ¿Dan compatibles? Si no, ¿qué hipótesis del modelo está
fallando?

**3.12.** *(Informe 1)* Con todo lo de las Clases 1 a 3: reportá el período del faro con el mejor
método disponible, justificá la elección del método con el gráfico de la Sección 6, y discutí qué
error limitaría el resultado si midieras una tanda de 500 destellos en lugar de 50.